# Модуль 1. Введение в сбор данных

- [ ] Соберать данные из двух разных источников (открытый датасет + веб-скрейпинг или API).
- [ ] Провести их агрегацию, создав единый датасет.
- [ ] Провести разведывательный анализ данных (EDA).
- [ ]  Постройть базовые визуализации для основных признаков с учетом разметки данных.
- [ ] Описать возможные применения этих данных в контексте машинного обучения.


# Описание датасета
### Идея:

Собрать новостные данные за определенный промежуток времени для получения кластеризации компаний по их наименованиям. Соответственно, решается 2 задачи. Первая - сбор новостных постов, вторая - извлечение названий компаний. В рамках PoC решения будут собраны только новостные посты для их дальнейшей кластеризации.

### Реализация:

  - Первой задачей (решаемая сейчас) будет парсинг новостных Telegram-каналов для получения постов, обогащение датасета открытыми датасетами.
  - Второй задачей (MVP-итерация) - разметка названий упоминаемых компаний для решения NER на новостных постах, обогащение открытыми датасетами

### Применение для машинного обучения:

Кластеризации компаний по их наименованиям для поиска инфо-поводов.

### Алгоритм обработки:

Объект класса ProductScraper в зависимости от переданного класса магазина, посылает запросы на сайт с помощью Selenium, парсит результаты через BautifulSoup по найденным в коде страницы идентификаторам для поиска нужных элементов. Данные считываются постранично с проверкой существования в csv такого же товара за эту же дату. С сайтов получаем информацию: название товара, цена, скидка, рейтинг. (возможно дальнейшее добавление)

Считанная (новая) информация с текущей датой дозаписывается в csv с названием обрабатываемого магазина.

В дальнейшем производится извлечение дополнительных признаков из поля name:

бренд
вес и единица измерения
в нарезке (да/нет)
БЗМЖ (да/нет)
Далее данные с извлеченными фичами записываются в БД. Обработка пропусков планируется в ЛР 2.

Замечания:

Скрейпинг магазинов потребовал дополнительных шагов: для Пятерочки это отправка предварительного запроса на авторизацию, для Магнита - отправка cookies с адресом магазина.

In [ ]:
import asyncio
import os
import pandas as pd
import datetime
import random
import time
from telethon import TelegramClient
from telethon.errors import SessionPasswordNeededError, FloodWaitError, RpcError
from telethon.tl.functions.messages import GetHistoryRequest
from dotenv import load_dotenv

# Загрузка переменных окружения для API Telegram
load_dotenv()

# Параметры для подключения к API Telegram
api_id = os.getenv('TELEGRAM_API_ID')
api_hash = os.getenv('TELEGRAM_API_HASH')
phone = os.getenv('TELEGRAM_PHONE')
username = os.getenv('TELEGRAM_USERNAME')

# Список каналов для парсинга
channels = ['forbesrussia', 'rt_russian', 'vedomosti', 'kommersant', 'rbcnews']

# Функция для получения сообщений из канала с защитой от блокировки
async def get_messages_from_channel(client, channel, limit_date):
    """
    Получает сообщения из указанного канала начиная с указанной даты
    с защитой от блокировки API
    
    Args:
        client: Экземпляр TelegramClient
        channel: Имя канала
        limit_date: Дата, начиная с которой нужно получить сообщения
        
    Returns:
        list: Список словарей с сообщениями
    """
    try:
        entity = await client.get_entity(channel)
        
        messages = []
        offset_id = 0
        limit = 50  # Уменьшенный лимит запросов
        total_messages = 0
        
        while True:
            try:
                # Случайная задержка между запросами для имитации человеческого поведения
                await asyncio.sleep(random.uniform(2.0, 5.0))
                
                history = await client(GetHistoryRequest(
                    peer=entity,
                    offset_id=offset_id,
                    offset_date=None,
                    add_offset=0,
                    limit=limit,
                    max_id=0,
                    min_id=0,
                    hash=0
                ))
                
                if not history.messages:
                    break
                    
                for message in history.messages:
                    if message.date < limit_date:
                        # Прекращаем сбор, если сообщение старше указанной даты
                        return messages
                        
                    # Добавляем только сообщения с текстом
                    if message.message:
                        messages.append({
                            'id': message.id,
                            'date': message.date,
                            'text': message.message,
                            'channel': channel,
                            'views': getattr(message, 'views', 0),
                            'forwards': getattr(message, 'forwards', 0)
                        })
                        
                offset_id = history.messages[-1].id
                total_messages += len(history.messages)
                
                # Если собрали много сообщений, делаем более длительную паузу
                if total_messages % 200 == 0:
                    print(f"Собрано {total_messages} сообщений из канала {channel}. Делаем паузу...")
                    await asyncio.sleep(random.uniform(10.0, 15.0))
                
            except FloodWaitError as e:
                # Если получили ошибку о превышении лимита, ждем указанное время
                print(f"Достигнут лимит запросов. Ожидание {e.seconds} секунд...")
                await asyncio.sleep(e.seconds + random.uniform(1.0, 5.0))
                continue
            except Exception as e:
                print(f"Ошибка при получении сообщений из канала {channel}: {e}")
                # Делаем паузу перед повторной попыткой
                await asyncio.sleep(random.uniform(5.0, 10.0))
                break
                
        return messages
    except Exception as e:
        print(f"Не удалось получить сообщения из канала {channel}: {e}")
        return []

# Основная функция для парсинга всех каналов
async def parse_telegram_channels(days_ago=7):
    """
    Парсит сообщения из указанных каналов за последние N дней
    
    Args:
        days_ago: Количество дней назад для ограничения выборки
        
    Returns:
        DataFrame: Датафрейм с собранными сообщениями
    """
    # Создаем клиент и входим в аккаунт
    client = TelegramClient(username, api_id, api_hash)
    await client.start()
    
    # Проверяем, авторизован ли клиент
    if not await client.is_user_authorized():
        await client.send_code_request(phone)
        try:
            await client.sign_in(phone, input('Введите код подтверждения: '))
        except SessionPasswordNeededError:
            await client.sign_in(password=input('Введите двухфакторный пароль: '))
    
    # Определяем дату, с которой нужно получать сообщения
    limit_date = datetime.datetime.now() - datetime.timedelta(days=days_ago)
    
    # Последовательно получаем сообщения из каналов с паузами между ними
    all_messages = []
    for channel in channels:
        print(f"Начинаем сбор данных из канала {channel}...")
        channel_messages = await get_messages_from_channel(client, channel, limit_date)
        all_messages.extend(channel_messages)
        print(f"Собрано {len(channel_messages)} сообщений из канала {channel}")
        
        # Делаем паузу между обработкой каналов
        await asyncio.sleep(random.uniform(15.0, 30.0))
    
    # Создаем DataFrame
    df = pd.DataFrame(all_messages)
    
    # Закрываем соединение
    await client.disconnect()
    
    return df

# Функция для запуска парсинга
def run_telegram_parser(days_ago=7):
    """
    Запускает парсинг Telegram-каналов
    
    Args:
        days_ago: Количество дней назад для ограничения выборки
        
    Returns:
        DataFrame: Датафрейм с собранными сообщениями
    """
    loop = asyncio.get_event_loop()
    df = loop.run_until_complete(parse_telegram_channels(days_ago))
    return df

# Пример использования
# Получаем новости за последние 7 дней
print("Начинаем сбор новостных данных из Telegram...")
telegram_news_df = run_telegram_parser(days_ago=7)

# Выводим информацию о собранных данных
print(f"Собрано {len(telegram_news_df)} новостных постов из {len(channels)} каналов")
print(f"Распределение постов по каналам:\n{telegram_news_df['channel'].value_counts()}")

# Сохраняем данные в CSV
telegram_news_df.to_csv('telegram_news_data.csv', index=False)

# Показываем первые несколько строк датасета
telegram_news_df.head()


: 